# 03 · Demo: trợ lý hỏi đáp pháp luật (link công khai)

Chạy toàn bộ hệ thống trên GPU của Kaggle: hybrid retrieval → reranker → cổng từ chối → Qwen2.5-3B + LoRA v2,
rồi mở giao diện Gradio qua một **link công khai** `https://….gradio.live` để dùng thử và quay demo.

**Trước khi chạy:**
1. **Settings:** Accelerator **GPU T4** (x1 là đủ) · Internet **On**.
2. **Add Input:** dataset chứa adapter LoRA v2 (các file `adapter_model.safetensors`, `adapter_config.json`, ...).
   Không bắt buộc: dataset chứa sẵn index (`dense.faiss`, `bm25_matrix.npz`, ...) để bỏ qua bước build khoảng 20 phút.
3. Chạy **tay từng ô** (không dùng Save & Run All): demo chỉ sống khi session đang mở.

Dùng xong nhớ **Stop session** để không tốn quota GPU.

In [ ]:
REPO_URL = "https://github.com/sinhtruc24/Vietnamese-Legal-RAG.git"
WORK = "/kaggle/working/Vietnamese-Legal-RAG"

import os, socket
try:
    socket.create_connection(("github.com", 443), timeout=5).close()
except OSError:
    raise SystemExit("Không có Internet: Settings -> Internet = On, rồi Restart và chạy lại.")
if not os.path.exists(WORK):
    !git clone -q {REPO_URL} {WORK}
else:
    !git -C {WORK} pull -q
%cd {WORK}
!git log --oneline -1

In [ ]:
!pip install -q -e ".[ml]" gradio
!pip install -q -r requirements-train.txt
!pip uninstall -y -q torchao   # torchao cũ của Kaggle làm peft lỗi khi gắn LoRA

## 1. Dữ liệu và index
Nếu Input có sẵn index thì dùng luôn; nếu không thì build (BM25 khoảng 1 phút, dense bge-m3 khoảng 20 phút trên T4).
Muốn khởi động nhanh hơn có thể đặt `RETRIEVER = "bm25"` (bỏ dense; vẫn có reranker).

In [ ]:
import glob
RETRIEVER = "hybrid"   # hybrid | bm25

uploaded = sorted(glob.glob("/kaggle/input/**/dense.faiss", recursive=True))
if uploaded:
    INDEX_DIR = os.path.dirname(uploaded[0])
    print("Dùng index có sẵn:", INDEX_DIR)
else:
    INDEX_DIR = "indexes/full"
    !python scripts/download_data.py
    if RETRIEVER == "bm25":
        !python scripts/build_index.py --bigrams --skip-dense --index-dir {INDEX_DIR}
    else:
        !python scripts/build_index.py --bigrams --index-dir {INDEX_DIR} --batch-size 64
print(sorted(os.listdir(INDEX_DIR)))

## 2. Adapter LoRA

In [ ]:
adapters = sorted(glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True),
                  key=lambda p: ("lora_v2" not in p, p))   # ưu tiên bản v2 nếu có nhiều bản
assert adapters, "Chưa có adapter: Add Input -> dataset chứa adapter_model.safetensors"
ADAPTER = os.path.dirname(adapters[0])
print("Adapter:", ADAPTER, sorted(os.listdir(ADAPTER)))

## 3. Chạy demo
Ô dưới chạy liên tục (không kết thúc) và in ra dòng **`Running on public URL: https://….gradio.live`**:
mở link đó trên trình duyệt. Nạp model mất khoảng 2–3 phút. Bấm ■ (Interrupt) để dừng demo.

In [ ]:
os.environ.update({
    "INDEX_DIR": INDEX_DIR,
    "RETRIEVER": RETRIEVER,
    "USE_RERANKER": "true",
    "CANDIDATES": "20",
    "REFUSAL_THRESHOLD": "0.67",       # cổng từ chối, chọn trên validation (results/refusal_gate_keep0.85.json)
    "LLM_BACKEND": "hf",               # chạy transformers + LoRA ngay trong tiến trình, không cần vLLM
    "LLM_MODEL": "Qwen/Qwen2.5-3B-Instruct",
    "LLM_ADAPTER": ADAPTER,
})
!python app/ui.py --local --share